# Databricks Lakehouse - Silver Layer Data Cleansing & Quarantine (`02_transform_silver`)

## Overview
This notebook processes raw data from the **Bronze Layer** (`workspace.neuro_insight.bronze_telemetry`) into the **Silver Layer**. It performs data validation against realistic domain rules, routes validated clean records into the Silver Delta table (`workspace.neuro_insight.silver_telemetry`), and quarantines malformed or out-of-range records into a dedicated quarantine table (`workspace.neuro_insight.silver_quarantine`).

---

### Step 1: Environment Setup & Bronze Data Ingestion
Import PySpark functions and load raw patient telemetry records from the Bronze Delta table.

In [ ]:
from pyspark.sql import functions as F

# Read raw telemetry data from Bronze table
bronze_df = spark.table("workspace.neuro_insight.bronze_telemetry")

print("Total Bronze records loaded:", bronze_df.count())

### Step 2: Define Data Quality Validation Rules
Apply validation flags (`is_valid`) to evaluate record integrity based on key clinical range checks and null-value constraints[cite: 2]:
- `patient_id`, `device_id`, and `event_timestamp` must not be null[cite: 2].
- `tremor_score` must fall between 0 and 10[cite: 2].
- `reaction_time_ms` and `gait_speed_mps` must be strictly positive values[cite: 2].
- `cognitive_score` must fall between 0 and 100[cite: 2].
- `heart_rate_bpm` must be within a physiologically plausible range (30 to 220 BPM)[cite: 2].

In [ ]:
validated_df = (
    bronze_df
    .withColumn(
        "is_valid",
        (
            F.col("patient_id").isNotNull()
            & F.col("device_id").isNotNull()
            & F.col("event_timestamp").isNotNull()
            & F.col("tremor_score").between(0, 10)
            & (F.col("reaction_time_ms") > 0)
            & (F.col("gait_speed_mps") > 0)
            & F.col("cognitive_score").between(0, 100)
            & F.col("heart_rate_bpm").between(30, 220)
        )
    )
)

### Step 3: Extract & Save Clean Silver Records
Filter records where `is_valid == True`, attach processing audit metadata (`processed_timestamp`), drop internal validation flags, and persist to the Silver Delta table[cite: 2].

In [ ]:
# Filter valid records and enrich with processing timestamp
silver_df = (
    validated_df
    .filter(F.col("is_valid") == True)
    .drop("is_valid")
    .withColumn("processed_timestamp", F.current_timestamp())
)

print("Clean Silver records count:", silver_df.count())

# Write clean dataset to Silver Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.neuro_insight.silver_telemetry")
)

### Step 4: Extract & Quarantine Invalid Records
Filter records failing validation (`is_valid == False`), enrich with audit timestamp (`quarantine_timestamp`), and route them to a dedicated quarantine table for debugging and remediation[cite: 2].

In [ ]:
# Filter failed/corrupt records and tag quarantine metadata
quarantine_df = (
    validated_df
    .filter(F.col("is_valid") == False)
    .drop("is_valid")
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

print("Quarantined records count:", quarantine_df.count())

# Write quarantined dataset to Quarantine Delta table
(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.neuro_insight.silver_quarantine")
)

print("Silver and quarantine tables created successfully.")